# Hyperparameter Tuning

This notebook tunes the hyperparameters for LightGBM, XGBoost, and the neural network. It follows the same structure as the comparison notebook: imports, settings, predictor definitions, helper functions, data loading, target creation, feature construction, model pipelines, search, and output.

## Imports

All needed packages are imported here. Seeds and TensorFlow settings are set at the top to make repeated runs as reproducible as possible.

In [ ]:
# =========================================================
# 0. IMPORTS
# =========================================================

import os
import json
import random
import warnings
import io
import contextlib
from copy import deepcopy

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# GLOBAL SEED / DETERMINISM
# ----------------------------------------------------------

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# ----------------------------------------------------------
# CORE
# ----------------------------------------------------------

import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)

# ----------------------------------------------------------
# METRICS AND VALIDATION
# ----------------------------------------------------------

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold

# ----------------------------------------------------------
# MODELS
# ----------------------------------------------------------

import lightgbm as lgb
from xgboost import XGBRegressor

# ----------------------------------------------------------
# PREPROCESSING
# ----------------------------------------------------------

from sklearn.preprocessing import StandardScaler

# ----------------------------------------------------------
# NEURAL NETS
# ----------------------------------------------------------

import tensorflow as tf

tf.get_logger().setLevel("ERROR")
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

## Global Settings

The main settings define the target, models, validation setup, search size, and export paths.

In [ ]:
# =========================================================
# 1. GLOBAL SETTINGS
# =========================================================

# time resolution and target variable
time_resolution = "qh"   # "qh" or "h"
target_col = "id1"       # "id1" or "id3"

# models used in the tuning search
selected_models = ["lgb", "xgb", "nn"]

# first year reserved as holdout period; all earlier years are used for tuning
test_year = 2025

# validation settings
valid_fraction = 0.15
cv_n_splits = 4
cv_shuffle = True

# evolutionary search settings
search_rounds = 3
keep_top_n = 5
mutation_probability = 0.35
random_injection_share = 0.25

initial_random_candidates = {
    "lgb": 18,
    "xgb": 18,
    "nn": 10,
}

offspring_candidates_per_round = {
    "lgb": 14,
    "xgb": 14,
    "nn": 8,
}

# lag settings
fundamental_lags = [1, 2]

print("selected models:", selected_models)
print("cv splits:", cv_n_splits)
print("search rounds:", search_rounds)

selected models: ['lgb', 'xgb', 'nn']
cv splits: 4
search rounds: 3


## Predictor Definitions

The model uses current predictor variables, lagged versions of selected fundamentals, and time-based variables.

In [3]:
# =========================================================
# 2. PREDICTOR DEFINITIONS
# =========================================================

# current, non-lagged predictors used by the model
predictor_vars = [
    "day_ahead_price",
    "load_actual_mw",
    "load_delta",
    "wind_delta",
    "solar_delta",
    "import_delta",
    "export_delta",
    "net_import_total",
    "fossil_gas_mw",
    "biomass_mw",
    "hydro_run_of_river_and_poundage_mw",
    "hydro_water_reservoir_mw",
    "hydro_pumped_storage_mw",
    "solar_mw",
    "wind_onshore_mw",
    "outage_total_true",
    "ramp_outage",
]

# subset of predictors for which lagged versions are added
lagged_fundamental_vars = [
    "load_delta",
    "wind_delta",
    "solar_delta",
    "net_import_total",
    "ramp_outage",
]

time_features = [
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "free_day",
]

## Helper Functions

These functions create lagged features, build the final feature list, split validation data, apply bias correction, and calculate metrics.

In [4]:
# =========================================================
# 3. HELPER FUNCTIONS
# =========================================================

def add_lag_features(df, columns, lags):
    df = df.copy()

    for col in columns:
        if col not in df.columns:
            continue

        for lag in lags:
            df[f"{col}_lag{lag}"] = df[col].shift(lag)

    return df


def build_feature_columns(df, predictor_vars, lagged_fundamental_vars, fundamental_lags, time_features):
    current_features = [col for col in predictor_vars if col in df.columns]

    lagged_features = [
        f"{col}_lag{lag}"
        for col in lagged_fundamental_vars
        for lag in fundamental_lags
        if f"{col}_lag{lag}" in df.columns
    ]

    available_time_features = [col for col in time_features if col in df.columns]

    feature_cols = current_features + lagged_features + available_time_features
    return list(dict.fromkeys(feature_cols))


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate_price_metrics(actual_price, pred_price):
    residuals = np.asarray(actual_price) - np.asarray(pred_price)
    abs_residuals = np.abs(residuals)

    return {
        "MAE": float(mean_absolute_error(actual_price, pred_price)),
        "RMSE": float(rmse(actual_price, pred_price)),
        "R2": float(r2_score(actual_price, pred_price)),
        "ResidualMean": float(np.mean(residuals)),
        "p95_abs_residual": float(np.percentile(abs_residuals, 95)),
        "p99_abs_residual": float(np.percentile(abs_residuals, 99)),
    }


def score_delta_predictions(pred_delta, da_price, actual_price):
    pred_price = np.asarray(da_price) + np.asarray(pred_delta)
    return evaluate_price_metrics(actual_price, pred_price)


def make_random_validation_split(X_train, y_train, valid_fraction=0.15, seed=SEED):
    n_valid = int(np.floor(len(X_train) * valid_fraction))

    if n_valid < 1:
        raise ValueError("Validation set would be empty. Increase training data or valid_fraction.")

    rng = np.random.RandomState(seed)
    val_idx = rng.choice(X_train.index.to_numpy(), size=n_valid, replace=False)

    X_val = X_train.loc[val_idx].copy()
    y_val = y_train.loc[val_idx].copy()
    X_tr = X_train.drop(val_idx).copy()
    y_tr = y_train.drop(val_idx).copy()

    return X_tr, y_tr, X_val, y_val


def compute_mean_bias(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) - np.asarray(y_pred)))


def apply_bias_correction(y_pred, bias_correction):
    return np.asarray(y_pred) + bias_correction


def lgb_eval_metric_from_objective(objective):
    if objective == "mae":
        return "l1"
    return "l2"


def nn_loss_from_name(loss_name):
    if loss_name == "mae":
        return "mae"
    if loss_name == "mse":
        return "mse"
    if loss_name == "huber":
        return tf.keras.losses.Huber()

    raise ValueError(f"Unsupported NN loss_name: {loss_name}")


@contextlib.contextmanager
def suppress_training_output():
    """Suppress model training output while keeping outer notebook prints visible."""
    with contextlib.redirect_stdout(io.StringIO()):
        with contextlib.redirect_stderr(io.StringIO()):
            yield

## Load Data

The combined dataset is loaded and sorted by datetime.

In [5]:
# =========================================================
# 4. LOAD DATA
# =========================================================

df = pd.read_csv(
    f"../data/combined_data_{time_resolution}.csv",
    parse_dates=["datetime"],
)

df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_convert("Europe/Vienna")
df = df.sort_values("datetime").reset_index(drop=True)
df = df.loc[:, ~df.columns.duplicated()].copy()

print("data loaded:", df.shape)
print("start:", df["datetime"].min())
print("end:  ", df["datetime"].max())

data loaded: (140256, 132)
start: 2022-01-01 00:00:00+01:00
end:   2025-12-31 23:45:00+01:00


## Create Target and Lagged Features

The model target is the intraday spread over the day-ahead price. Lagged features are created from the selected fundamental variables.

In [6]:
# =========================================================
# 5. CREATE TARGET AND LAGGED FEATURES
# =========================================================

spread_col = f"spread_{target_col}"
df[spread_col] = df[target_col] - df["day_ahead_price"]

df = add_lag_features(df, lagged_fundamental_vars, fundamental_lags)

print("target:", spread_col)
print("lags:  ", fundamental_lags)

target: spread_id1
lags:   [1, 2]


## Build Feature List

The final feature list combines current predictors, lagged predictors, and time-based predictors.

In [7]:
# =========================================================
# 6. BUILD FEATURE LIST
# =========================================================

feature_cols = build_feature_columns(
    df=df,
    predictor_vars=predictor_vars,
    lagged_fundamental_vars=lagged_fundamental_vars,
    fundamental_lags=fundamental_lags,
    time_features=time_features,
)

print(f"n_features = {len(feature_cols)}")
print(feature_cols)

n_features = 32
['day_ahead_price', 'load_actual_mw', 'load_delta', 'wind_delta', 'solar_delta', 'import_delta', 'export_delta', 'net_import_total', 'fossil_gas_mw', 'biomass_mw', 'hydro_run_of_river_and_poundage_mw', 'hydro_water_reservoir_mw', 'hydro_pumped_storage_mw', 'solar_mw', 'wind_onshore_mw', 'outage_total_true', 'ramp_outage', 'load_delta_lag1', 'load_delta_lag2', 'wind_delta_lag1', 'wind_delta_lag2', 'solar_delta_lag1', 'solar_delta_lag2', 'net_import_total_lag1', 'net_import_total_lag2', 'ramp_outage_lag1', 'ramp_outage_lag2', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'free_day']


## Build Modeling DataFrame

The modeling dataframe is created and split into the tuning period and the final holdout test year.

In [8]:
# =========================================================
# 7. BUILD MODELING DATAFRAME AND TRAIN/TEST SPLIT
# =========================================================

needed_cols = ["datetime", target_col, "day_ahead_price", spread_col] + feature_cols
needed_cols = list(dict.fromkeys([col for col in needed_cols if col in df.columns]))

df_model = df[needed_cols].dropna().copy()

train_df = df_model[df_model["datetime"].dt.year < test_year].copy()
test_df = df_model[df_model["datetime"].dt.year == test_year].copy()

print("df_model shape:", df_model.shape)
print("train_df shape:", train_df.shape)
print("test_df shape: ", test_df.shape)
print("train period:", train_df["datetime"].min(), "->", train_df["datetime"].max())
print("test period: ", test_df["datetime"].min(), "->", test_df["datetime"].max())

df_model shape: (139772, 35)
train_df shape: (104828, 35)
test_df shape:  (34944, 35)
train period: 2022-01-01 01:00:00+01:00 -> 2024-12-31 23:45:00+01:00
test period:  2025-01-01 00:00:00+01:00 -> 2025-12-31 23:45:00+01:00


## Baseline Parameters and Search Space

The baseline parameters are included in every search. The search spaces define allowed values for random sampling, recombination, and mutation.

In [9]:
# =========================================================
# 8. BASELINE PARAMS AND SEARCH SPACE
# =========================================================

baseline_params = {
    "lgb": {
        "objective": "mae",
        "eval_metric": "l1",
        "n_estimators": 400,
        "learning_rate": 0.05,
        "max_depth": 5,
        "num_leaves": 24,
        "min_child_samples": 40,
        "subsample": 0.7,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.5,
        "reg_lambda": 0.5,
    },
    "xgb": {
        "objective": "reg:absoluteerror",
        "eval_metric": "mae",
        "n_estimators": 400,
        "learning_rate": 0.05,
        "max_depth": 5,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "gamma": 1.0,
        "reg_alpha": 0.5,
        "reg_lambda": 2.0,
    },
    "nn": {
        "loss_name": "mae",
        "learning_rate": 1e-3,
        "units_1": 64,
        "units_2": 32,
        "dropout_1": 0.25,
        "dropout_2": 0.15,
        "l2_reg": 1e-4,
        "epochs": 70,
        "batch_size": 256,
    },
}

search_spaces = {
    "lgb": {
        "objective": ["mae"],
        "eval_metric": ["l1"],
        "n_estimators": [400],
        "learning_rate": [0.02, 0.04, 0.05, 0.06, 0.08],
        "max_depth": [4, 5, 6],
        "num_leaves": [15, 23, 31, 47],
        "min_child_samples": [30, 40, 60, 100],
        "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.7, 0.8, 0.9],
        "reg_alpha": [0.0, 0.5, 2.0, 10.0],
        "reg_lambda": [0.5, 2.0, 10.0],
    },
    "xgb": {
        "objective": ["reg:absoluteerror"],
        "eval_metric": ["mae"],
        "n_estimators": [400],
        "learning_rate": [0.02, 0.04, 0.05, 0.06, 0.08],
        "max_depth": [4, 5, 6],
        "min_child_weight": [4, 5, 8, 15],
        "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.7, 0.8, 0.9],
        "gamma": [0.0, 0.5, 1.5, 5.0],
        "reg_alpha": [0.0, 0.5, 2.0, 5.0],
        "reg_lambda": [1.0, 2.0, 10.0, 20.0],
    },
    "nn": {
        "loss_name": ["mae"],
        "learning_rate": [5e-4, 1e-3, 2e-3],
        "units_1": [48, 64, 96, 128],
        "units_2": [24, 32, 48, 64],
        "dropout_1": [0.15, 0.25, 0.35, 0.45],
        "dropout_2": [0.05, 0.15, 0.25],
        "l2_reg": [1e-5, 1e-4, 1e-3],
        "epochs": [70],
        "batch_size": [128, 256],
    },
}

## Evolutionary Proposal Functions

The first round samples random candidates. Later rounds recombine parameter values from the best previous candidates and add random mutations. A small share of fresh random candidates is kept to avoid narrowing the search too early.

In [ ]:
# =========================================================
# 9. EVOLUTIONARY PROPOSAL FUNCTIONS
# =========================================================

def candidate_key(params):
    # Convert parameter dictionary to a unique, sorted JSON string for duplicate checking
    return json.dumps(params, sort_keys=True)


def sample_random_candidate(model_name):
    # Generate a completely random set of hyperparameters from the defined search space
    return {
        key: random.choice(values)
        for key, values in search_spaces[model_name].items()
    }


def mutate_value(current_value, allowed_values):
    # If only one option exists, return it immediately
    if len(allowed_values) == 1:
        return allowed_values[0]

    # Fallback to random choice if current value is not in the allowed list
    if current_value not in allowed_values:
        return random.choice(allowed_values)

    # Determine neighbor indices for small local steps in the parameter space
    idx = allowed_values.index(current_value)
    possible_steps = [-1, 1]

    # Allow larger jumps if the parameter has many possible values
    if len(allowed_values) >= 5:
        possible_steps = [-2, -1, 1, 2]

    # Pick a random step and ensure the new index stays within valid bounds
    step = random.choice(possible_steps)
    new_idx = max(0, min(len(allowed_values) - 1, idx + step))
    return allowed_values[new_idx]


def recombine_candidate(model_name, parent_pool, mutation_probability=mutation_probability):
    # Get the search space and initialize a new empty child candidate
    space = search_spaces[model_name]
    child = {}

    # Inherit each hyperparameter from a random parent in the pool
    for key, allowed_values in space.items():
        parent = random.choice(parent_pool)
        value = parent.get(key, random.choice(allowed_values))

        # Occasionally apply a mutation to introduce fresh variation
        if random.random() < mutation_probability:
            value = mutate_value(value, allowed_values)

        child[key] = value

    return child


def build_initial_candidates(model_name, n_random):
    # Start the first generation with a known baseline configuration
    candidates = [("baseline", deepcopy(baseline_params[model_name]))]
    seen = {candidate_key(candidates[0][1])}

    # Fill the rest of the initial population with unique random configurations
    while len(candidates) < n_random + 1:
        cand = sample_random_candidate(model_name)
        key = candidate_key(cand)

        if key not in seen:
            candidates.append(("random", cand))
            seen.add(key)

    return candidates


def build_evolution_candidates(model_name, parent_pool, n_candidates):
    # Initialize the next generation's pool and a set to track uniqueness
    candidates = []
    seen = set()

    # Calculate how many candidates will be new random injections vs. bred children
    n_random = int(round(n_candidates * random_injection_share))
    n_children = n_candidates - n_random

    # Produce new child candidates through recombination until the quota is met
    while len([x for x in candidates if x[0] == "child"]) < n_children:
        cand = recombine_candidate(model_name, parent_pool)
        key = candidate_key(cand)

        if key not in seen:
            candidates.append(("child", cand))
            seen.add(key)

    # Inject completely random candidates to maintain genetic diversity
    while len([x for x in candidates if x[0] == "random"]) < n_random:
        cand = sample_random_candidate(model_name)
        key = candidate_key(cand)

        if key not in seen:
            candidates.append(("random", cand))
            seen.add(key)

    return candidates

## Model Pipelines

Each model pipeline trains one candidate configuration and returns bias-corrected spread predictions for the validation fold.

In [11]:
# =========================================================
# 10. MODEL PIPELINES
# =========================================================

def run_lgb_pipeline_params(X_train, y_train, X_test, y_test, params, seed=SEED, valid_fraction=valid_fraction):
    X_tr, y_tr, X_val, y_val = make_random_validation_split(
        X_train,
        y_train,
        valid_fraction=valid_fraction,
        seed=seed,
    )

    model = lgb.LGBMRegressor(
        objective=params["objective"],
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        num_leaves=params["num_leaves"],
        min_child_samples=params["min_child_samples"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],
        random_state=seed,
        n_jobs=-1,
    )

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric=lgb_eval_metric_from_objective(params["objective"]),
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )

    y_val_pred_raw = model.predict(X_val)
    bias_correction = compute_mean_bias(y_val, y_val_pred_raw)

    y_pred_raw = model.predict(X_test)
    y_pred = apply_bias_correction(y_pred_raw, bias_correction)

    return y_pred, bias_correction


def run_xgb_pipeline_params(X_train, y_train, X_test, y_test, params, seed=SEED, valid_fraction=valid_fraction):
    X_tr, y_tr, X_val, y_val = make_random_validation_split(
        X_train,
        y_train,
        valid_fraction=valid_fraction,
        seed=seed,
    )

    model = XGBRegressor(
        objective=params["objective"],
        eval_metric=params["eval_metric"],
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        min_child_weight=params["min_child_weight"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        gamma=params["gamma"],
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],
        random_state=seed,
        n_jobs=-1,
        tree_method="hist",
    )

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    y_val_pred_raw = model.predict(X_val)
    bias_correction = compute_mean_bias(y_val, y_val_pred_raw)

    y_pred_raw = model.predict(X_test)
    y_pred = apply_bias_correction(y_pred_raw, bias_correction)

    return y_pred, bias_correction


def run_nn_pipeline_params(X_train, y_train, X_test, y_test, params, seed=SEED, valid_fraction=valid_fraction):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.keras.backend.clear_session()

    X_tr, y_tr, X_val, y_val = make_random_validation_split(
        X_train,
        y_train,
        valid_fraction=valid_fraction,
        seed=seed,
    )

    scaler = StandardScaler()

    X_tr_s = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    model = Sequential([
        Input(shape=(X_tr_s.shape[1],)),
        Dense(params["units_1"], activation="relu", kernel_regularizer=l2(params["l2_reg"])),
        Dropout(params["dropout_1"]),
        Dense(params["units_2"], activation="relu", kernel_regularizer=l2(params["l2_reg"])),
        Dropout(params["dropout_2"]),
        Dense(1),
    ])

    model.compile(
        optimizer=Adam(learning_rate=params["learning_rate"]),
        loss=nn_loss_from_name(params["loss_name"]),
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True,
        )
    ]

    model.fit(
        X_tr_s,
        y_tr,
        validation_data=(X_val_s, y_val),
        epochs=params["epochs"],
        batch_size=params["batch_size"],
        verbose=0,
        callbacks=callbacks,
        shuffle=False,
    )

    y_val_pred_raw = model.predict(X_val_s, verbose=0).ravel()
    bias_correction = compute_mean_bias(y_val, y_val_pred_raw)

    y_pred_raw = model.predict(X_test_s, verbose=0).ravel()
    y_pred = apply_bias_correction(y_pred_raw, bias_correction)

    return y_pred, bias_correction

## Cross-Validated Evaluation

Each candidate is evaluated on fixed cross-validation folds. Inside each fold, the model pipeline still uses an internal validation split for early stopping and bias correction.

In [12]:
# =========================================================
# 11. CROSS-VALIDATED EVALUATION
# =========================================================

def get_cv_data(train_df, feature_cols):
    X = train_df[feature_cols].reset_index(drop=True).copy()
    y = train_df[spread_col].reset_index(drop=True).copy()
    actual_price = train_df[target_col].to_numpy()
    da_price = train_df["day_ahead_price"].to_numpy()

    return X, y, actual_price, da_price


def make_cv_splits(X, cv_splits=cv_n_splits, cv_shuffle=cv_shuffle, seed=SEED):
    kf = KFold(
        n_splits=cv_splits,
        shuffle=cv_shuffle,
        random_state=seed,
    )

    return list(kf.split(X))


def evaluate_params_cv(model_name, params, train_df, feature_cols, cv_splits=cv_n_splits, cv_shuffle=cv_shuffle, seed=SEED):
    X, y, actual_price, da_price = get_cv_data(train_df, feature_cols)
    splits = make_cv_splits(X, cv_splits=cv_splits, cv_shuffle=cv_shuffle, seed=seed)

    fold_metrics = []
    fold_biases = []

    for fold_idx, (tr_idx, va_idx) in enumerate(splits, start=1):
        X_tr = X.iloc[tr_idx].copy()
        y_tr = y.iloc[tr_idx].copy()
        X_va = X.iloc[va_idx].copy()
        y_va = y.iloc[va_idx].copy()

        seed_fold = seed + fold_idx

        if model_name == "lgb":
            pred_delta, bias = run_lgb_pipeline_params(
                X_train=X_tr,
                y_train=y_tr,
                X_test=X_va,
                y_test=y_va,
                params=params,
                seed=seed_fold,
                valid_fraction=valid_fraction,
            )
        elif model_name == "xgb":
            pred_delta, bias = run_xgb_pipeline_params(
                X_train=X_tr,
                y_train=y_tr,
                X_test=X_va,
                y_test=y_va,
                params=params,
                seed=seed_fold,
                valid_fraction=valid_fraction,
            )
        elif model_name == "nn":
            pred_delta, bias = run_nn_pipeline_params(
                X_train=X_tr,
                y_train=y_tr,
                X_test=X_va,
                y_test=y_va,
                params=params,
                seed=seed_fold,
                valid_fraction=valid_fraction,
            )
        else:
            raise ValueError(model_name)

        fold_metrics.append(
            score_delta_predictions(
                pred_delta=pred_delta,
                da_price=da_price[va_idx],
                actual_price=actual_price[va_idx],
            )
        )
        fold_biases.append(bias)

    return {
        "CV_MAE": float(np.mean([m["MAE"] for m in fold_metrics])),
        "CV_RMSE": float(np.mean([m["RMSE"] for m in fold_metrics])),
        "CV_R2": float(np.mean([m["R2"] for m in fold_metrics])),
        "CV_ResidualMean": float(np.mean([m["ResidualMean"] for m in fold_metrics])),
        "CV_p95_abs_residual": float(np.mean([m["p95_abs_residual"] for m in fold_metrics])),
        "CV_p99_abs_residual": float(np.mean([m["p99_abs_residual"] for m in fold_metrics])),
        "mean_bias_correction": float(np.mean(fold_biases)),
        "n_rows_used": int(len(X)),
    }

## Evolutionary Search Loop

The search starts with baseline plus random candidates. Later rounds keep the best candidates and generate new candidates by recombining their parameter values with random mutations.

In [ ]:
# =========================================================
# 12. EVOLUTIONARY SEARCH LOOP
# =========================================================

def evaluate_candidate(model_name, search_type, round_idx, trial_idx, params, train_df, feature_cols):
    # Execute cross-validation for a specific parameter set and collect performance metrics
    cv_metrics = evaluate_params_cv(
        model_name=model_name,
        params=params,
        train_df=train_df,
        feature_cols=feature_cols,
        cv_splits=cv_n_splits,
        cv_shuffle=cv_shuffle,
        seed=SEED,
    )

    # Return a consolidated dictionary containing metadata, metrics, and serialized parameters
    return {
        "Model": model_name,
        "SearchType": search_type,
        "Round": round_idx,
        "Trial": trial_idx,
        **cv_metrics,
        "params": json.dumps(params, sort_keys=True),
    }


def select_parent_pool(history_df, keep_top_n=keep_top_n):
    # Sort the global history by error metrics to identify the most successful candidates
    top_df = history_df.sort_values(["CV_MAE", "CV_RMSE"]).head(keep_top_n)
    # Deserialize the parameter strings back into dictionaries for the next breeding step
    return [json.loads(x) for x in top_df["params"].tolist()]


def run_evolutionary_search_for_model(model_name, train_df, feature_cols):
    # Initialize a list to store the evaluation results of all trials
    history = []

    # Iterate through the specified number of evolutionary generations
    for round_idx in range(1, search_rounds + 1):
        print(f"\n===== {model_name.upper()} | ROUND {round_idx}/{search_rounds} =====")

        # First round uses baseline and random seeds; subsequent rounds evolve from previous bests
        if round_idx == 1:
            candidate_specs = build_initial_candidates(
                model_name=model_name,
                n_random=initial_random_candidates[model_name],
            )
        else:
            # Select successful parents from history to generate the next population
            history_df_so_far = pd.DataFrame(history)
            parent_pool = select_parent_pool(history_df_so_far)

            candidate_specs = build_evolution_candidates(
                model_name=model_name,
                parent_pool=parent_pool,
                n_candidates=offspring_candidates_per_round[model_name],
            )

        round_rows = []

        # Evaluate each candidate in the current generation's pool
        for trial_idx, (search_type, params) in enumerate(candidate_specs, start=1):
            print(
                f"{model_name.upper()} round {round_idx} "
                f"trial {trial_idx}/{len(candidate_specs)} [{search_type}]"
            )

            # Suppress verbose training logs to keep the console output clean
            with suppress_training_output():
                row = evaluate_candidate(
                    model_name=model_name,
                    search_type=search_type,
                    round_idx=round_idx,
                    trial_idx=trial_idx,
                    params=params,
                    train_df=train_df,
                    feature_cols=feature_cols,
                )

            # Log results to both the global history and the current round tracker
            history.append(row)
            round_rows.append(row)

        # Summary output: show the top 5 performers of the current round
        round_df = pd.DataFrame(round_rows).sort_values(["CV_MAE", "CV_RMSE"]).reset_index(drop=True)

        print(
            round_df[
                [
                    "Model",
                    "SearchType",
                    "Round",
                    "Trial",
                    "CV_MAE",
                    "CV_RMSE",
                    "CV_R2",
                    "CV_p95_abs_residual",
                    "CV_p99_abs_residual",
                ]
            ].head(5).to_string(index=False)
        )

    # Compile the full search history and extract the best parameters found across all rounds
    history_df = pd.DataFrame(history).sort_values(["CV_MAE", "CV_RMSE"]).reset_index(drop=True)
    best_params = json.loads(history_df.iloc[0]["params"])

    return history_df, best_params

## Run Search

The evolutionary search is executed for all selected model types.

In [14]:
# =========================================================
# 13. RUN SEARCH FOR ALL MODELS
# =========================================================

search_histories = {}
best_params_by_model = {}

for model_name in selected_models:
    history_df, best_params = run_evolutionary_search_for_model(
        model_name=model_name,
        train_df=train_df,
        feature_cols=feature_cols,
    )

    search_histories[model_name] = history_df
    best_params_by_model[model_name] = best_params

    print(f"\n===== BEST {model_name.upper()} PARAMS =====")
    print(json.dumps(best_params, indent=4, sort_keys=True))


===== LGB | ROUND 1/3 =====
LGB round 1 trial 1/19 [baseline]
LGB round 1 trial 2/19 [random]
LGB round 1 trial 3/19 [random]
LGB round 1 trial 4/19 [random]
LGB round 1 trial 5/19 [random]
LGB round 1 trial 6/19 [random]
LGB round 1 trial 7/19 [random]
LGB round 1 trial 8/19 [random]
LGB round 1 trial 9/19 [random]
LGB round 1 trial 10/19 [random]
LGB round 1 trial 11/19 [random]
LGB round 1 trial 12/19 [random]
LGB round 1 trial 13/19 [random]
LGB round 1 trial 14/19 [random]
LGB round 1 trial 15/19 [random]
LGB round 1 trial 16/19 [random]
LGB round 1 trial 17/19 [random]
LGB round 1 trial 18/19 [random]
LGB round 1 trial 19/19 [random]
Model SearchType  Round  Trial    CV_MAE   CV_RMSE    CV_R2  CV_p95_abs_residual  CV_p99_abs_residual
  lgb     random      1     10 34.294244 72.302715 0.725803           106.383596           227.027672
  lgb     random      1     18 34.388036 72.175692 0.726786           106.702497           226.322056
  lgb     random      1      6 34.783077 72.7

## Search Result Tables

The best candidates are printed for each model. The ranking is based on cross-validated MAE, with RMSE used as a secondary sorting metric.

In [15]:
# =========================================================
# 14. SEARCH RESULT TABLES
# =========================================================

result_cols = [
    "Model",
    "SearchType",
    "Round",
    "Trial",
    "CV_MAE",
    "CV_RMSE",
    "CV_R2",
    "CV_ResidualMean",
    "CV_p95_abs_residual",
    "CV_p99_abs_residual",
    "mean_bias_correction",
    "n_rows_used",
]

for model_name in selected_models:
    print("\n" + "=" * 100)
    print(f"TOP RESULTS: {model_name.upper()}")
    print("=" * 100)

    print(
        search_histories[model_name][result_cols]
        .head(10)
        .to_string(index=False)
    )


TOP RESULTS: LGB
Model SearchType  Round  Trial    CV_MAE   CV_RMSE    CV_R2  CV_ResidualMean  CV_p95_abs_residual  CV_p99_abs_residual  mean_bias_correction  n_rows_used
  lgb     random      1     10 34.294244 72.302715 0.725803        -0.500033           106.383596           227.027672             -1.036334       104828
  lgb      child      3      9 34.333136 72.233829 0.726327        -0.515728           106.373500           226.726509             -1.075788       104828
  lgb     random      1     18 34.388036 72.175692 0.726786        -0.508724           106.702497           226.322056             -1.123026       104828
  lgb      child      3      2 34.715805 72.574164 0.723772        -0.571242           107.449776           228.207901             -1.137537       104828
  lgb      child      2      2 34.740152 72.620264 0.723448        -0.509606           107.174800           228.117375             -1.157421       104828
  lgb      child      2     10 34.745941 72.688355 0.72289

## Best Parameter Dictionaries

The final best parameter dictionaries are printed so they can be copied directly into the comparison notebook if needed.

In [17]:
# =========================================================
# 16. PRINT BEST PARAMETER DICTIONARIES
# =========================================================

for model_name in selected_models:
    print("\n" + "=" * 100)
    print(f"BEST PARAMS: {model_name.upper()}")
    print("=" * 100)
    print(json.dumps(best_params_by_model[model_name], indent=4, sort_keys=True))


BEST PARAMS: LGB
{
    "colsample_bytree": 0.8,
    "eval_metric": "l1",
    "learning_rate": 0.08,
    "max_depth": 6,
    "min_child_samples": 40,
    "n_estimators": 400,
    "num_leaves": 31,
    "objective": "mae",
    "reg_alpha": 10.0,
    "reg_lambda": 10.0,
    "subsample": 0.9
}

BEST PARAMS: XGB
{
    "colsample_bytree": 0.9,
    "eval_metric": "mae",
    "gamma": 0.5,
    "learning_rate": 0.08,
    "max_depth": 6,
    "min_child_weight": 4,
    "n_estimators": 400,
    "objective": "reg:absoluteerror",
    "reg_alpha": 0.0,
    "reg_lambda": 10.0,
    "subsample": 0.8
}

BEST PARAMS: NN
{
    "batch_size": 128,
    "dropout_1": 0.25,
    "dropout_2": 0.15,
    "epochs": 70,
    "l2_reg": 1e-05,
    "learning_rate": 0.002,
    "loss_name": "mae",
    "units_1": 128,
    "units_2": 64
}
